# Błędne jednostki w `VALUE`

**Pytanie:** czemu część wierszy ma cenę domniemaną około 1000 razy niższą od ceny rynkowej i czy można to skorygować?

**Wniosek:** część filerów nadal podaje `VALUE` w tysiącach, mimo zmiany SEC z 3 I 2023. Błąd dotyczy prawie zawsze całego raportu, więc korekta działa na poziomie raportu. Decyzja i progi: `docs/decisions.md`.

Dane: `positions_pit.parquet`. Kolumna `value_usd` to wartość przed korektą. Raporty mieszane są już z tego pliku usunięte, ich liczba jest w `docs/data_note.md`.

In [1]:
%load_ext sql
%sql duckdb://
%config SqlMagic.displaylimit = 20

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'duckdb://'

Odniesieniem jest mediana ceny dla papierów z co najmniej 20 posiadaczami.

In [2]:
%%sql
CREATE OR REPLACE VIEW lines AS
WITH p AS (
    SELECT * FROM '../data/processed/positions_pit.parquet'
    WHERE shares > 0 AND value_usd > 0
),
x AS (
    SELECT CUSIP, PERIODOFREPORT, median(value_usd * 1.0 / shares) AS price
    FROM p GROUP BY CUSIP, PERIODOFREPORT HAVING count(*) >= 20
)
SELECT p.*, p.value_usd * 1.0 / p.shares / x.price AS ratio
FROM p JOIN x ON p.CUSIP = x.CUSIP AND p.PERIODOFREPORT = x.PERIODOFREPORT;

CREATE OR REPLACE VIEW reports AS
SELECT CIK, PERIODOFREPORT, count(*) AS n, avg((ratio BETWEEN 0.0005 AND 0.002)::INT) AS low_share,
       sum(value_usd) AS raw_value
FROM lines GROUP BY CIK, PERIODOFREPORT;

Running query in 'duckdb://'

Count


## 1. Czy błąd dotyczy pojedynczych wierszy, czy całych raportów?

Prawie wszystkie zaniżone wiersze są w raportach, które są zaniżone niemal w całości.

In [3]:
%%sql
SELECT CASE WHEN low_share >= 0.9 THEN 'co najmniej 90%'
            WHEN low_share >= 0.1 THEN '10-90%'
            WHEN low_share > 0 THEN 'poniżej 10%'
            ELSE 'brak' END AS zanizone_wiersze,
       count(*) AS raporty, sum(n) AS wiersze
FROM reports GROUP BY 1 ORDER BY 2

Running query in 'duckdb://'

zanizone_wiersze,raporty,wiersze
10-90%,6,17
poniżej 10%,333,304719
co najmniej 90%,8424,2660895
brak,103402,24985728


## 2. Czy to się zmienia w czasie?

Udział spada od pierwszego kwartału po zmianie reguły, co pasuje do filerów stopniowo przechodzących na dolary.

In [4]:
%%sql
SELECT PERIODOFREPORT, count(*) AS raporty,
       sum((low_share >= 0.9)::INT) AS w_tysiacach,
       round(100.0 * sum((low_share >= 0.9)::INT) / count(*), 1) AS procent
FROM reports GROUP BY 1 ORDER BY 1

Running query in 'duckdb://'

PERIODOFREPORT,raporty,w_tysiacach,procent
2022-12-31,6691,1172,17.5
2023-03-31,6665,818,12.3
2023-06-30,6711,726,10.8
2023-09-30,6725,665,9.9
2023-12-31,7012,614,8.8
2024-03-31,7079,587,8.3
2024-06-30,7128,542,7.6
2024-09-30,7179,518,7.2
2024-12-31,7761,494,6.4
2025-03-31,7837,454,5.8


## 3. Czy korekta razy 1000 daje właściwą cenę?

Po przemnożeniu stosunek do ceny rynkowej wraca w okolice 1.

In [5]:
%%sql
SELECT count(*) AS wiersze,
       round(median(l.ratio * 1000), 3) AS mediana_stosunku,
       round(100.0 * sum((l.ratio * 1000 BETWEEN 0.9 AND 1.1)::INT) / count(*), 1) AS procent_w_10pct
FROM lines l JOIN reports r ON l.CIK = r.CIK AND l.PERIODOFREPORT = r.PERIODOFREPORT
WHERE r.low_share >= 0.9

Running query in 'duckdb://'

wiersze,mediana_stosunku,procent_w_10pct
2660895,1.0,96.3


## 4. Czemu nie wystarczy pominąć tego w etykiecie?

Filerzy zmieniają jednostki między kwartałami, co bez korekty zniekształca cechy oparte na wartości.

In [6]:
%%sql
SELECT prev_thousands, now_thousands, count(*) AS przejscia
FROM (
    SELECT low_share >= 0.9 AS now_thousands,
           lag(low_share >= 0.9) OVER (PARTITION BY CIK ORDER BY PERIODOFREPORT) AS prev_thousands
    FROM reports
)
WHERE prev_thousands IS NOT NULL GROUP BY 1, 2 ORDER BY 1, 2

Running query in 'duckdb://'

prev_thousands,now_thousands,przejscia
False,False,93624
False,True,142
True,False,957
True,True,7001


## 5. Czy `TABLEVALUETOTAL` pomaga wykryć błąd?

Nie: filerzy w tysiącach podają też sumę w tysiącach, więc stosunek wynosi około 1 w obu grupach.

In [7]:
%%sql
WITH totals AS (
    SELECT lpad(s.CIK, 10, '0') AS CIK, s.PERIODOFREPORT, sum(t.TABLEVALUETOTAL) AS total
    FROM '../data/processed/submission/*.parquet' s
    JOIN '../data/processed/summarypage/*.parquet' t ON s.ACCESSION_NUMBER = t.ACCESSION_NUMBER
    WHERE s.SUBMISSIONTYPE = '13F-HR' AND s.FILING_DATE <= s.PERIODOFREPORT + INTERVAL 50 DAY
    GROUP BY 1, 2
),
values_ AS (
    SELECT CIK, PERIODOFREPORT, sum(value_usd) AS value_sum
    FROM '../data/processed/positions_pit.parquet' GROUP BY 1, 2
)
SELECT r.low_share >= 0.9 AS w_tysiacach, count(*) AS raporty,
       round(median(v.value_sum * 1.0 / nullif(t.total, 0)), 3) AS mediana_stosunku
FROM reports r
JOIN values_ v ON r.CIK = v.CIK AND r.PERIODOFREPORT = v.PERIODOFREPORT
JOIN totals t ON r.CIK = t.CIK AND r.PERIODOFREPORT = t.PERIODOFREPORT
GROUP BY 1 ORDER BY 1

Running query in 'duckdb://'

w_tysiacach,raporty,mediana_stosunku
False,103740,1.0
True,8424,1.0
